In [9]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df_train = pd.read_csv("train.csv/train.csv")
df_test = pd.read_csv("test.csv/test.csv")
df_test_labels = pd.read_csv("test_labels.csv/test_labels.csv")
df_sample_submission = pd.read_csv("sample_submission.csv/sample_submission.csv")

df_train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,"Explanation\r\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27",0,0,0,0,0,0
1,000103f0d9cfb60f,"D'aww! He matches this background colour I'm seemingly stuck with. Thanks. (talk) 21:51, January 11, 2016 (UTC)",0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It's just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page. He seems to care more about the formatting than the actual info.",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions on improvement - I wondered if the section statistics should be later on, or a subsection of """"types of accidents"""" -I think the references may need tidying so that they are all in the exact same format ie date format etc. I can do that later on, if no-one else does first - if you have any preferences for formatting style on references or want to do it yourself please let me know.\r\n\r\nThere appears to be a backlog on articles for review so I guess there may be a delay until a reviewer turns up. It's listed in the relevant form eg Wikipedia:Good_article_nominations#Transport """,0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember what page that's on?",0,0,0,0,0,0


In [10]:
import html
import re

def preprocess_text(text):
    """Clean a comment while keeping wording, negation, and punctuation."""
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " EMAIL ", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " IPADDRESS ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

# Keep the original text; use clean_text as the model's text input.
df_train["clean_text"] = df_train["comment_text"].apply(preprocess_text)
df_test["clean_text"] = df_test["comment_text"].apply(preprocess_text)

df_train[["comment_text", "clean_text"]].head()

,comment_text,clean_text
0,"Explanation\r\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27","explanation why the edits made under my username hardcore metallica fan were reverted? they weren't vandalisms, just closure on some gas after i voted at new york dolls fac. and please don't remove the template from the talk page since i'm retired now. ipaddress"
1,"D'aww! He matches this background colour I'm seemingly stuck with. Thanks. (talk) 21:51, January 11, 2016 (UTC)","d'aww! he matches this background colour i'm seemingly stuck with. thanks. (talk) 21:51, january 11, 2016 (utc)"
2,"Hey man, I'm really not trying to edit war. It's just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page. He seems to care more about the formatting than the actual info.","hey man, i'm really not trying to edit war. it's just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page. he seems to care more about the formatting than the actual info."
3,"""\r\nMore\r\nI can't make any real suggestions on improvement - I wondered if the section statistics should be later on, or a subsection of """"types of accidents"""" -I think the references may need tidying so that they are all in the exact same format ie date format etc. I can do that later on, if no-one else does first - if you have any preferences for formatting style on references or want to do it yourself please let me know.\r\n\r\nThere appears to be a backlog on articles for review so I guess there may be a delay until a reviewer turns up. It's listed in the relevant form eg Wikipedia:Good_article_nominations#Transport """,""" more i can't make any real suggestions on improvement - i wondered if the section statistics should be later on, or a subsection of """"types of accidents"""" -i think the references may need tidying so that they are all in the exact same format ie date format etc. i can do that later on, if no-one else does first - if you have any preferences for formatting style on references or want to do it yourself please let me know. there appears to be a backlog on articles for review so i guess there may be a delay until a reviewer turns up. it's listed in the relevant form eg wikipedia:good_article_nominations#transport """
4,"You, sir, are my hero. Any chance you remember what page that's on?","you, sir, are my hero. any chance you remember what page that's on?"


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Select the comment text as the input features for the model.
X = df_train["clean_text"]

# Select the binary toxicity label that the model will learn to predict.
# A value of 0 means non-toxic, and a value of 1 means toxic.
y = df_train["toxic"]

# Split the labeled data into a training set and a validation set.
# The model learns from 80% of the comments and is evaluated on the other 20%.
# A fixed random_state makes the split reproducible.
# Stratification keeps the toxic/non-toxic proportions similar in both sets.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create a Bag of Words vectorizer that counts words in each comment.
# It learns the vocabulary only from training comments to prevent data leakage.
bow_vectorizer = CountVectorizer()

# Learn the vocabulary and convert each training comment into a word-count vector.
X_train_bow = bow_vectorizer.fit_transform(X_train)

# Convert validation comments using the same vocabulary learned from training.
X_valid_bow = bow_vectorizer.transform(X_valid)

# Show the number of comments and vocabulary terms in each matrix.
print("Training matrix:", X_train_bow.shape)
print("Validation matrix:", X_valid_bow.shape)

# Report how many unique terms the vectorizer learned from training comments.
print("Vocabulary size:", len(bow_vectorizer.get_feature_names_out()))

Training matrix: (127656, 156585)
Validation matrix: (31915, 156585)
Vocabulary size: 156585


In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# Train a Naive Bayes classifier using the Bag of Words training matrix.
bow_model = MultinomialNB()
bow_model.fit(X_train_bow, y_train)

# Predict the toxicity labels for comments kept out of training.
y_valid_pred = bow_model.predict(X_valid_bow)

# Report precision, recall, and F1-score for both classes.
print(classification_report(
    y_valid,
    y_valid_pred,
    target_names=["Non-toxic", "Toxic"],
    zero_division=0
))

# Rows are the actual labels and columns are the predicted labels.
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_valid, y_valid_pred, labels=[0, 1]))

              precision    recall  f1-score   support

   Non-toxic       0.96      0.98      0.97     28856
       Toxic       0.78      0.64      0.70      3059

    accuracy                           0.95     31915
   macro avg       0.87      0.81      0.84     31915
weighted avg       0.95      0.95      0.95     31915

Confusion matrix (rows = actual, columns = predicted):
[[28299   557]
 [ 1093  1966]]


In [13]:
import pandas as pd

preds = pd.read_csv("preds/bert_test.csv")
comments = pd.read_csv("test.csv/test.csv")

result = comments[["id", "comment_text"]].merge(preds, on="id")
result = result.sort_values("toxic", ascending=False)

result[["comment_text", "toxic", "severe_toxic", "obscene", "threat"]].head(10)

comment_text  \
44563                                                                                                                                                                  pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n\r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n pelican shit you fuckers! \r\n 

In [20]:
import pandas as pd

# Choisis le fichier de prédictions à consulter
#predictions = pd.read_csv("preds/bert_test.csv")
# Pour le modèle TF-IDF, utilise plutôt :
predictions = pd.read_csv("preds/tfidf_lr_test.csv")

comments = pd.read_csv("out/test_clean.csv")

# Associer chaque prédiction à son commentaire d'origine
result = predictions.merge(
    comments[["id", "comment_text"]],
    on="id",
    how="left"
)

# Remplace cette valeur par l'id que tu veux retrouver
comment_id = "03cfa8a23b91b009"

result.loc[
    result["id"] == comment_id,
    ["id", "comment_text", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
]

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
914,03cfa8a23b91b009,We might just all be dicks trying to look serious when talking about two girls shitting in a cup and eating chocolate.,0.814522,0.00551,0.232456,0.000958,0.040855,0.003418


In [21]:
import pandas as pd
pd.read_csv("preds/bert_val.csv").shape[0]

500